In [62]:
!pip install sqlalchemy
!pip install SQLAlchemy
import requests
import time
import datetime
import os
from sqlalchemy import create_engine, Table, Column, Integer, Float, String, MetaData, DateTime
from sqlalchemy.orm import sessionmaker
from pymongo import MongoClient
import pandas as pd



In [63]:
# ---------- Configuration ----------

API_KEY = 'xxxxxxxx'  # Polygon API Key

# Define all currency pairs to be monitored
CURRENCY_PAIRS = [pair.strip() for pair in [
    'BTCUSD', 'USDEUR', 'USDCAD', 'USDGBP', 'USDCHF', 'USDAUD',
    'EURCHF', 'GBPEUR', 'GBPCHF', 'USDJPY', 'USDINR', 'USDCNY'
]]

# Separate base currency pairs (for feature generation) from the remaining ones
base_cps = CURRENCY_PAIRS[1:6]
remaining_cps = CURRENCY_PAIRS[6:]

TOTAL_PERIODS = 20           # Total number of collection rounds
INTERVAL_SECONDS = 6 * 60    # Interval between each collection (6 minutes)

# ---------- SQLite Setup ----------

# Connect to SQLite database
engine = create_engine('sqlite:///final_fx.db', connect_args={'timeout': 30})
metadata = MetaData()

# Define auxiliary table to store raw FX rates
aux_table = Table('aux_fx_rates', metadata,
    Column('id', Integer, primary_key=True),
    Column('currency_pair', String),
    Column('rate_value', Float),
    Column('rate_timestamp', DateTime),
    Column('entry_timestamp', DateTime)
)

# Define final statistics table to store aggregated results
final_table = Table('final_fx_stats', metadata,
    Column('id', Integer, primary_key=True),
    Column('currency_pair', String),
    Column('data_timestamp', DateTime),
    Column('db_timestamp', DateTime),
    Column('max', Float),
    Column('min', Float),
    Column('mean', Float),
    Column('vol', Float),
    Column('fd', Float)
)

# Create the tables in SQLite database
metadata.create_all(engine)

# Create a session object for SQLite database operations
Session = sessionmaker(bind=engine)

# ---------- MongoDB Setup ----------

# Connect to local MongoDB instance
client = MongoClient('mongodb://localhost:27017/')
mongo_db = client['fx_data_hw2']

# Define MongoDB collections (analogous to the SQLite tables)
mongo_aux = mongo_db['aux_fx_rates']
mongo_final = mongo_db['final_fx_stats']

# ---------- Utility Functions ----------

def fetch_fx_rate(pair):
    """
    Fetch real-time FX rate from Polygon API for a given currency pair.
    Returns a dictionary with pair, rate, and timestamps.
    """
    url = f'https://api.polygon.io/v1/conversion/{pair[:3]}/{pair[3:]}'
    params = {'amount': 1, 'precision': 4, 'apiKey': API_KEY}
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        rate = data['converted']
        timestamp = datetime.datetime.now()
        return {
            'currency_pair': pair,
            'rate_value': rate,
            'rate_timestamp': timestamp,
            'entry_timestamp': timestamp
        }
    return None

def compute_stats(data_list):
    """
    Compute max, min, mean, and volatility from a list of FX rate entries.
    """
    rates = [d['rate_value'] for d in data_list]
    if not rates:
        return None
    max_r = max(rates)
    min_r = min(rates)
    mean_r = sum(rates) / len(rates)
    vol = (max_r - min_r) / mean_r if mean_r else 0
    return max_r, min_r, mean_r, vol

def compute_keltner(mean):
    """
    Generate upper and lower Keltner bands based on mean price.
    Returns a list of (lower, upper) tuples for n = 1 to 1000.
    """
    return [(mean - n * 0.00001 * mean, mean + n * 0.00001 * mean) for n in range(1, 1001)]

def export_to_csv(row, filename="final_output_.csv"):
    """
    Append a dictionary row to a CSV file.
    If the file does not exist, create it and write the header.
    """
    df = pd.DataFrame([row])
    df.to_csv(filename, index=False, mode='a', header=not os.path.exists(filename))

# ---------- Keltner Band Intersection Counter ----------

def find_keltner_band_intersections(fx_rates, k_upper_bands, k_lower_bands):
    """
    Count how many times FX prices cross Keltner bands.
    Also tracks the highest band crossed on both upper and lower sides.
    
    Returns:
        num_intersections: total number of crossings
        peak_upper_intersection: highest upper band crossed
        peak_lower_intersection: highest lower band crossed
    """
    num_intersections = 0
    upper_intersections = []  # Store indexes of crossed upper bands
    lower_intersections = []  # Store indexes of crossed lower bands
    prev_up_i = 0             # Previous upper band index
    prev_dw_i = 0             # Previous lower band index
    prev_switch = None        # Track direction of last cross: 'up' or 'down'

    for fx_rate in fx_rates:
        # Check if rate crosses upper band
        if fx_rate >= k_upper_bands[0]:
            for i in range(len(k_upper_bands) - 1):
                if k_upper_bands[i] <= fx_rate < k_upper_bands[i + 1]:
                    if i != prev_up_i:
                        if prev_switch == 'down':
                            # Switching from down to up: record all past intersections
                            for j in range(prev_dw_i)[::-1]:
                                num_intersections += 1
                                lower_intersections.append(j)
                            for j in range(i + 1):
                                num_intersections += 1
                                upper_intersections.append(j)
                        else:
                            # Same direction (up): accumulate crossings
                            if i > prev_up_i:
                                for j in range(prev_up_i + 1, i + 1):
                                    num_intersections += 1
                                    upper_intersections.append(j)
                            elif i < prev_up_i:
                                for j in range(i, prev_up_i)[::-1]:
                                    num_intersections += 1
                                    upper_intersections.append(j)
                        prev_up_i = i
                        prev_switch = 'up'
                        prev_dw_i = 0
                        continue

        # Check if rate crosses lower band
        if fx_rate <= k_lower_bands[0]:
            for i in range(len(k_lower_bands) - 1):
                if k_lower_bands[i] >= fx_rate > k_lower_bands[i + 1]:
                    if i != prev_dw_i:
                        if prev_switch == 'up':
                            # Switching from up to down: record all past intersections
                            for j in range(prev_up_i)[::-1]:
                                num_intersections +=_


In [70]:
# ---------- Main Logic ----------

# Dictionary to hold previous period means for Keltner calculation
previous_means = {}

# Dictionary to track the maximum number of intersections (used for FD normalization)
max_fd_dict = {pair: 1 for pair in CURRENCY_PAIRS}

# Loop over each of the total collection periods (20 times)
for period in range(1, TOTAL_PERIODS + 1):
    print(f"\n[⏱️] Period {period} started at {datetime.datetime.now().strftime('%H:%M:%S')}")

    session = Session()  # Create a new database session

    try:
        # Clear previous period data from both SQLite and MongoDB aux tables
        session.execute(aux_table.delete())
        mongo_aux.delete_many({})

        # Initialize an empty buffer dictionary to collect data for each currency pair
        buffer = {pair: [] for pair in CURRENCY_PAIRS}
        start_time = datetime.datetime.now()  # Record period start time

        # Collect data continuously for the duration of INTERVAL_SECONDS (6 minutes)
        while (datetime.datetime.now() - start_time).seconds < INTERVAL_SECONDS:
            for pair in CURRENCY_PAIRS:
                record = fetch_fx_rate(pair)  # Fetch real-time FX rate
                if record:
                    buffer[pair].append(record)  # Store in memory buffer
                    session.execute(aux_table.insert().values(**record))  # Insert into SQLite
                    mongo_aux.insert_one(record)  # Insert into MongoDB
            time.sleep(5)  # Sleep to avoid hitting API rate limits

        # After data collection, process each currency pair
        for pair in CURRENCY_PAIRS:
            data = buffer[pair]  # Retrieve all collected data for this pair

            stats = compute_stats(data)  # Compute max, min, mean, vol
            if not stats:
                continue  # Skip if no data available

            rates = [d['rate_value'] for d in data]
            mean = stats[2]  # Mean value for this period

            # Calculate FD (Fractal Dimension) based on Keltner band intersections
            if period > 1:
                # Use the mean from the previous period to compute bands
                k_bands = compute_keltner(previous_means[pair])
                k_lower_bands = [b[0] for b in k_bands]
                k_upper_bands = [b[1] for b in k_bands]

                # Count how many times this period's rates cross Keltner bands
                N, peak_upper, peak_lower = find_keltner_band_intersections(rates, k_upper_bands, k_lower_bands)

                # Update max intersection count for normalization
                max_fd_dict[pair] = max(max_fd_dict[pair], N)

                # Compute normalized FD value
                fd = N / max_fd_dict[pair] if max_fd_dict[pair] else 0.0
            else:
                fd = 0.0  # No FD for the first period

            # Construct the final statistics row for this pair
            row = {
                'currency_pair': pair,
                'data_timestamp': start_time,                    # When this batch started
                'db_timestamp': datetime.datetime.now(),         # When this row is saved
                'max': stats[0],
                'min': stats[1],
                'mean': stats[2],
                'vol': stats[3],
                'fd': fd
            }

            # Save the row to SQLite, MongoDB, and CSV
            session.execute(final_table.insert().values(**row))
            session.commit()
            mongo_final.insert_one(row)
            export_to_csv(row)

            # Save the mean for use in the next period's Keltner computation
            previous_means[pair] = mean

    finally:
        session.close()  # Ensure session is closed regardless of success/failure

print("✅ All 20 periods completed.")  # Final completion message



[⏱️] Period 1 started at 19:53:44

[⏱️] Period 2 started at 19:59:44

[⏱️] Period 3 started at 20:05:49

[⏱️] Period 4 started at 20:11:50

[⏱️] Period 5 started at 20:17:52

[⏱️] Period 6 started at 20:23:55

[⏱️] Period 7 started at 20:29:59

[⏱️] Period 8 started at 20:36:04

[⏱️] Period 9 started at 20:42:11

[⏱️] Period 10 started at 20:48:17

[⏱️] Period 11 started at 20:54:20

[⏱️] Period 12 started at 21:00:24

[⏱️] Period 13 started at 21:06:30

[⏱️] Period 14 started at 21:12:31

[⏱️] Period 15 started at 21:18:36

[⏱️] Period 16 started at 21:24:43

[⏱️] Period 17 started at 21:30:45

[⏱️] Period 18 started at 21:36:47

[⏱️] Period 19 started at 21:42:55

[⏱️] Period 20 started at 21:48:57
✅ All 20 periods completed.


In [74]:
!pip install arcticdb

In [122]:
import pandas as pd
import arcticdb as adb

# === 1. Load data from CSV ===
# Load previously saved final output data from CSV file
df = pd.read_csv("final_output_.csv")

# Convert the 'data_timestamp' column to datetime objects for time-based operations
df['data_timestamp'] = pd.to_datetime(df['data_timestamp'])

# === 2. Assign hour groups (hour 0, hour 1, etc.) ===
# We assume 20 records per currency_pair: the first 10 belong to hour 0, the next 10 to hour 1
# 'cumcount' assigns a 0-based index to each row within its currency_pair group
df['hour_group'] = df.groupby('currency_pair').cumcount() // 10

# === 3. Aggregate statistics per hour per currency_pair ===
# Compute the average of each metric (mean, max, min, vol, fd) for every currency_pair-hour_group
agg_df = df.groupby(['currency_pair', 'hour_group'])[['mean', 'max', 'min', 'vol', 'fd']].mean().rename(
    columns={
        'mean': 'average_mean',
        'max': 'average_max',
        'min': 'average_min',
        'vol': 'average_vol',
        'fd': 'average_fd'
    }
).reset_index()

# === 4. Add the earliest timestamp per hour group as representative timestamp ===
# This helps associate each group with a specific point in time (the start of that hour window)
min_time = df.groupby(['currency_pair', 'hour_group'])['data_timestamp'].min().reset_index(name='timestamp')

# Merge the aggregated stats with the corresponding timestamps
agg_df = pd.merge(agg_df, min_time, on=['currency_pair', 'hour_group'])

# === 5. Rearrange column order for clarity ===
agg_df = agg_df[['currency_pair', 'timestamp', 'average_mean', 'average_max', 'average_min', 'average_vol', 'average_fd']]

# === 6. Save the result into ArcticDB ===
# Connect to ArcticDB LMDB-based store
arctic = adb.Arctic("lmdb://arcticdb_fx_hourly_summary")

# Access (or create if needed) the library for hourly stats
lib = arctic.get_library("hourly_cp_stats", create_if_missing=True)

# Write the aggregated DataFrame to ArcticDB under the symbol "hourly_cp_summary"
lib.write("hourly_cp_summary", agg_df)


VersionedItem(symbol='hourly_cp_summary', library='hourly_cp_stats', data=n/a, version=1, metadata=None, host='LMDB(path=C:\\Users\\24716\\Final New\\Final on my own\\arcticdb_fx_hourly_summary)', timestamp=1746156295562942800)

In [128]:
# === 1. Connect to ArcticDB ===
# Initialize a connection to the ArcticDB instance (using LMDB backend)
arctic = adb.Arctic("lmdb://arcticdb_fx_hourly_summary")

# === 2. List all available libraries in the ArcticDB instance ===
print("📚 Libraries:")
print(arctic.list_libraries())  # Displays all logical groupings (similar to folders)

# === 3. Open a specific library and list all stored symbols ===
lib = arctic.get_library("hourly_cp_stats")  # Connect to the desired library
print("\n📦 Symbols in 'hourly_cp_stats':")
print(lib.list_symbols())  # Symbols are like individual time-series datasets or tables

# === 4. Read data from a specific symbol into a DataFrame ===
df_read = lib.read("hourly_cp_summary").data  # Load the symbol's data into a Pandas DataFrame

# === 5. Preview the loaded data ===
print("\n📊 First few rows of 'hourly_cp_summary':")
print(df_read.head())  # Show the top rows for inspection


📚 Libraries:
['hourly_cp_stats']

📦 Symbols in 'hourly_cp_stats':
['hourly_cp_summary']

📊 First few rows of 'hourly_cp_summary':
   currency_pair                  timestamp  average_mean  average_max  \
0         EURCHF 2025-05-01 19:53:44.127245      0.936937      0.93710   
1         EURCHF 2025-05-01 20:54:20.636208      0.937580      0.93773   
2         GBPCHF 2025-05-01 19:53:44.127245      1.102532      1.10276   
3         GBPCHF 2025-05-01 20:54:20.636208      1.103428      1.10365   
4         GBPEUR 2025-05-01 19:53:44.127245      1.176685      1.17683   
5         GBPEUR 2025-05-01 20:54:20.636208      1.176880      1.17702   
6         USDAUD 2025-05-01 19:53:44.127245      1.563795      1.56447   
7         USDAUD 2025-05-01 20:54:20.636208      1.561307      1.56183   
8         USDCAD 2025-05-01 19:53:44.127245      1.384573      1.38476   
9         USDCAD 2025-05-01 20:54:20.636208      1.383958      1.38412   
10        USDCHF 2025-05-01 19:53:44.127245      0.83018

In [140]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# === Polygon API Configuration ===
API_KEY = "beBybSi8daPgsTp5yx5cHtHpYcrjp5Jq"
BASE_URL = "https://api.polygon.io/v2/aggs/ticker/X:BTCUSD/range/6/minute"

# === Start times for two hourly windows ===
start_times = [
    datetime.fromisoformat("2025-05-01 19:53:44.127245"),
    datetime.fromisoformat("2025-05-01 20:54:20.636208")
]

btc_data_all = []

# === Fetch BTCUSD data for each hour group ===
for i, start_time in enumerate(start_times):
    end_time = start_time + timedelta(minutes=60)
    url = f"{BASE_URL}/{start_time.date()}/{end_time.date()}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 5000,
        "apiKey": API_KEY
    }

    # Request BTC data and limit to first 10 records for each hour window
    r = requests.get(url, params=params)
    data = r.json().get("results", [])[:10]
    df = pd.DataFrame(data)

    # Compute mean of high/low prices for correlation
    df['mean'] = (df['h'] + df['l']) / 2
    df['hour_group'] = i
    btc_data_all.append(df[['mean', 'hour_group']])

# Combine the two periods of BTC mean data
btc_df = pd.concat(btc_data_all).reset_index(drop=True)

# === Load your final_output_.csv data ===
raw_df = pd.read_csv("final_output_.csv")
raw_df['data_timestamp'] = pd.to_datetime(raw_df['data_timestamp'])

# === Define the 11 currency pairs to evaluate ===
currency_pairs = ['USDEUR', 'USDCAD', 'USDGBP', 'USDCHF', 'USDAUD',
                  'EURCHF', 'GBPEUR', 'GBPCHF', 'USDJPY', 'USDINR', 'USDCNY']

# Filter and assign hour_group based on row position
raw_df = raw_df[raw_df['currency_pair'].isin(currency_pairs)].copy()
raw_df['hour_group'] = raw_df.groupby('currency_pair').cumcount() // 10

# === Define safe Pearson correlation calculation ===
def safe_pearson(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return 0  # Return 0 if one of the series is constant (std = 0)
    return np.corrcoef(x, y)[0, 1]

# === Calculate correlation for 11 pairs × 2 hours = 22 results ===
results = []
for cp in currency_pairs:
    for hour in [0, 1]:
        cp_means = raw_df[(raw_df['currency_pair'] == cp) & (raw_df['hour_group'] == hour)]['mean'].tolist()
        btc_means = btc_df[btc_df['hour_group'] == hour]['mean'].tolist()

        # Only calculate if both series have 10 points
        if len(cp_means) == 10 and len(btc_means) == 10:
            corr = safe_pearson(cp_means, btc_means)
        else:
            corr = 0  # Fallback if data is insufficient

        results.append({
            'currency_pair': cp,
            'hour_group': hour,
            'correlation_with_btc': corr
        })

# === Output the correlation results ===
cor_df = pd.DataFrame(results)
print(cor_df)


   currency_pair  hour_group  correlation_with_btc
0         USDEUR           0              0.847665
1         USDEUR           1             -0.658189
2         USDCAD           0             -0.540910
3         USDCAD           1             -0.587732
4         USDGBP           0              0.792381
5         USDGBP           1             -0.493066
6         USDCHF           0              0.892212
7         USDCHF           1             -0.518430
8         USDAUD           0             -0.807620
9         USDAUD           1              0.319161
10        EURCHF           0              0.914126
11        EURCHF           1             -0.102612
12        GBPEUR           0              0.842286
13        GBPEUR           1             -0.744814
14        GBPCHF           0              0.901853
15        GBPCHF           1             -0.531279
16        USDJPY           0              0.793048
17        USDJPY           1             -0.778937
18        USDINR           0   

In [142]:
import pandas as pd
from datetime import datetime
import arcticdb as adb

# === 1. Construct correlation data ===
# Manually create a DataFrame with currency pair correlations (precomputed earlier)
cor_df = pd.DataFrame({
    'currency_pair': ['USDEUR', 'USDEUR', 'USDCAD', 'USDCAD', 'USDGBP', 'USDGBP', 'USDCHF', 'USDCHF',
                      'USDAUD', 'USDAUD', 'EURCHF', 'EURCHF', 'GBPEUR', 'GBPEUR', 'GBPCHF', 'GBPCHF',
                      'USDJPY', 'USDJPY', 'USDINR', 'USDINR', 'USDCNY', 'USDCNY'],
    'hour_group': [0, 1] * 11,  # Repeat hour_group 0 and 1 for each currency pair
    'correlation_with_btc': [
        0.847665, -0.658189, -0.540910, -0.587732, 0.792381, -0.493066,
        0.892212, -0.518430, -0.807620, 0.319161, 0.914126, -0.102612,
        0.842286, -0.744814, 0.901853, -0.531279, 0.793048, -0.778937,
        -0.838363, -0.756416, 0.000000, 0.000000
    ]
})

# === 2. Map timestamps to each hour group ===
# Assign representative timestamps for hour group 0 and 1
timestamp_map = {
    0: datetime.fromisoformat("2025-05-01 19:53:44.127245"),
    1: datetime.fromisoformat("2025-05-01 20:54:20.636208")
}
cor_df['timestamp'] = cor_df['hour_group'].map(timestamp_map)

# Keep only necessary columns
cor_df = cor_df[['currency_pair', 'timestamp', 'correlation_with_btc']]

# === 3. Load ArcticDB library ===
# Connect to the ArcticDB environment where your FX data is stored
arctic = adb.Arctic("lmdb://arcticdb_fx_hourly_summary")
lib = arctic.get_library("hourly_cp_stats")

# === 4. Read existing hourly summary data ===
# This symbol contains average values for each currency pair at each hourly window
existing_df = lib.read("hourly_cp_summary").data

# === 5. Merge BTC correlation into the hourly summary ===
# Join the correlation values into the summary dataframe using currency_pair and timestamp
merged_df = pd.merge(existing_df, cor_df, on=['currency_pair', 'timestamp'], how='left')

# === 6. Write the updated data back into ArcticDB ===
# This overwrites the original symbol with the enriched dataset
lib.write("hourly_cp_summary", merged_df)


VersionedItem(symbol='hourly_cp_summary', library='hourly_cp_stats', data=n/a, version=2, metadata=None, host='LMDB(path=C:\\Users\\24716\\Final New\\Final on my own\\arcticdb_fx_hourly_summary)', timestamp=1746157669946978300)

In [144]:
# === 1. Connect to ArcticDB ===
# Establish a connection to your ArcticDB instance using the LMDB backend
arctic = adb.Arctic("lmdb://arcticdb_fx_hourly_summary")

# === 2. List all available libraries ===
# Libraries in ArcticDB are similar to folders or namespaces
print("📚 Libraries:")
print(arctic.list_libraries())

# === 3. Open a specific library and list its symbols ===
# Each symbol inside a library represents a DataFrame/time series table
lib = arctic.get_library("hourly_cp_stats")
print("\n📦 Symbols in 'hourly_cp_stats':")
print(lib.list_symbols())

# === 4. Read data from a specific symbol as a DataFrame ===
# Load the contents of the symbol 'hourly_cp_summary'
df_read = lib.read("hourly_cp_summary").data

# === 5. Display the first few rows of the data ===
print("\n📊 First few rows of 'hourly_cp_summary':")
print(df_read)


📚 Libraries:
['hourly_cp_stats']

📦 Symbols in 'hourly_cp_stats':
['hourly_cp_summary']

📊 First few rows of 'hourly_cp_summary':
   currency_pair                  timestamp  average_mean  average_max  \
0         EURCHF 2025-05-01 19:53:44.127245      0.936937      0.93710   
1         EURCHF 2025-05-01 20:54:20.636208      0.937580      0.93773   
2         GBPCHF 2025-05-01 19:53:44.127245      1.102532      1.10276   
3         GBPCHF 2025-05-01 20:54:20.636208      1.103428      1.10365   
4         GBPEUR 2025-05-01 19:53:44.127245      1.176685      1.17683   
5         GBPEUR 2025-05-01 20:54:20.636208      1.176880      1.17702   
6         USDAUD 2025-05-01 19:53:44.127245      1.563795      1.56447   
7         USDAUD 2025-05-01 20:54:20.636208      1.561307      1.56183   
8         USDCAD 2025-05-01 19:53:44.127245      1.384573      1.38476   
9         USDCAD 2025-05-01 20:54:20.636208      1.383958      1.38412   
10        USDCHF 2025-05-01 19:53:44.127245      0.83018

In [1]:
import arcticdb as adb
import pandas as pd

# 1. Connect to ArcticDB
# Initialize the ArcticDB instance using LMDB as backend
arctic = adb.Arctic("lmdb://arcticdb_fx_hourly_summary")

# 2. Select the target library
# 'hourly_cp_stats' contains your hourly FX summaries
lib = arctic.get_library("hourly_cp_stats")

# 3. Read the data from the specified symbol
# 'hourly_cp_summary' holds the processed and enriched DataFrame
df = lib.read("hourly_cp_summary").data

# 4. Export the DataFrame to a CSV file
df.to_csv("hourly_cp_summary.csv", index=False)

print("✅ Data has been saved as hourly_cp_summary.csv")


✅ Data has been saved as hourly_cp_summary.csv
